# Capstone: Composed Agentic Workflow (Claude)

This capstone combines key patterns into one runnable system:
1. Planner step generation
2. ReAct-style tool use with `create_agent`
3. Reflection pass for quality
4. Memory using `thread_id` checkpointer state
5. Human-in-the-loop approval gate before execution

```mermaid
flowchart TD
    U[User Goal] --> P[Planner]
    P --> H{Approved?}
    H -->|No| Z[Stop]
    H -->|Yes| X[Executor with Tools + Memory]
    X --> C[Critic]
    C --> V[Reviser]
    V --> O[Final Answer]
```

In [ ]:
from datetime import datetime
from typing import TypedDict, List

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_anthropic import ChatAnthropic
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph

MODEL_NAME = "claude-haiku-4-5-20251001"
llm = ChatAnthropic(model=MODEL_NAME, temperature=0, max_tokens=1024)
memory = InMemorySaver()

In [ ]:
@tool
def current_time(_: str = "") -> str:
    """Return current local timestamp."""
    return datetime.now().isoformat(timespec="seconds")

@tool
def safe_calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression such as '(21*2)+7'."""
    allowed = set("0123456789+-*/(). %")
    if any(ch not in allowed for ch in expression):
        return "Only arithmetic characters are allowed."
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as exc:
        return f"Calculation error: {exc}"

In [ ]:
agent = create_agent(
    model=llm,
    tools=[current_time, safe_calculator],
    checkpointer=memory,
)

In [ ]:
class CapstoneState(TypedDict):
    user_goal: str
    approved: bool
    plan_steps: List[str]
    draft_answer: str
    critique: str
    final_answer: str

def planner_node(state: CapstoneState) -> CapstoneState:
    prompt = f"Create 3 concise steps to solve this goal:\n{state['user_goal']}\nReturn only numbered steps."
    raw = str(llm.invoke(prompt).content)
    lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]
    steps = []
    for line in lines:
        cleaned = line.lstrip('0123456789). -').strip()
        if cleaned:
            steps.append(cleaned)
    state['plan_steps'] = steps[:3] if steps else [
        "Understand the request",
        "Use tools for required facts/calculations",
        "Write a concise final response",
    ]
    return state

def approval_router(state: CapstoneState) -> str:
    return 'execute' if state['approved'] else 'stop'

def executor_node(state: CapstoneState) -> CapstoneState:
    cfg = {"configurable": {"thread_id": "capstone-demo"}}
    plan_text = "\n".join(f"{i+1}. {s}" for i, s in enumerate(state['plan_steps']))
    user_prompt = (
        f"Goal: {state['user_goal']}\n\n"
        f"Plan:\n{plan_text}\n\n"
        "Use tools when useful. Keep answer practical and concise."
    )
    result = agent.invoke(
        {"messages": [{"role": "user", "content": user_prompt}]},
        config=cfg,
    )
    state['draft_answer'] = str(result['messages'][-1].content)
    return state

def critic_node(state: CapstoneState) -> CapstoneState:
    critique_prompt = f"Critique this answer in 3 short bullets (correctness, clarity, completeness):\n{state['draft_answer']}"
    state['critique'] = str(llm.invoke(critique_prompt).content)
    return state

def reviser_node(state: CapstoneState) -> CapstoneState:
    revise_prompt = f"Revise the draft using the critique.\nDraft:\n{state['draft_answer']}\n\nCritique:\n{state['critique']}"
    state['final_answer'] = str(llm.invoke(revise_prompt).content)
    return state

In [ ]:
builder = StateGraph(CapstoneState)
builder.add_node('planner', planner_node)
builder.add_node('executor', executor_node)
builder.add_node('critic', critic_node)
builder.add_node('reviser', reviser_node)

builder.add_edge(START, 'planner')
builder.add_conditional_edges('planner', approval_router, {'execute': 'executor', 'stop': END})
builder.add_edge('executor', 'critic')
builder.add_edge('critic', 'reviser')
builder.add_edge('reviser', END)

graph = builder.compile()

In [ ]:
initial_state = {
    'user_goal': 'Compute (44*19)-13, include current time, and explain a simple 3-day plan to practice agentic AI.',
    'approved': True,
    'plan_steps': [],
    'draft_answer': '',
    'critique': '',
    'final_answer': '',
}

result = graph.invoke(initial_state)
result['final_answer']

## How the code cells map to the pattern
Cell 2 creates the Claude client and the shared checkpoint memory used by the composed workflow.
Cell 3 defines the tool functions that let the executor fetch time and perform safe calculations.
Cell 4 builds the ReAct agent that will run inside the executor branch and preserve state with the checkpointer.
Cell 5 defines the planner, approval gate, executor, critic, and reviser nodes that form the end-to-end graph.
Cell 6 compiles the graph wiring so the pattern becomes executable.
Cell 7 runs the approved example goal and returns the final refined answer.

## Human-in-the-loop toggle
Set `approved=False` in Cell 7 to stop the workflow before execution.

## Memory behavior
The executor uses `thread_id=capstone-demo`, so repeated runs can preserve conversational context across invocations.